# View Purge Behavior — full gate matrix

> **New here? Read this first.** A view drop's outcome depends on FOUR things, and
> this notebook maps how they interact — including isolating which one causes the
> `403` that an earlier session saw.

## The four dimensions

```
purge-view-metadata-on-drop   catalog config — should the drop delete the metadata file?
drop-with-purge.enabled       catalog config — is purge allowed at all?
?purgeRequested=              request param  — true / false / absent
CATALOG_MANAGE_CONTENT        privilege      — does the principal have purge rights?
```

In our no-STS environment, even a successful purging drop **orphans** the metadata
file (the async cleanup task fails — Apache Polaris #379, see the test report).

## The 8-case matrix

```
C1  meta=true  purge=false  grant=NO   req=absent   ← the original "403" scenario
C2  meta=true  purge=false  grant=YES  req=absent   isolates the PRIVILEGE gate
C3  meta=true  purge=true   grant=NO   req=absent   isolates the CONFIG gate
C4  meta=true  purge=true   grant=YES  req=absent   everything enabled
C5  meta=true  purge=true   grant=YES  req=false    explicit purgeRequested=false
C6  meta=false purge=true   grant=YES  req=true     param vs config conflict
C7  meta=false purge=true   grant=YES  req=absent   metadata-only baseline
C8  drop an already-dropped view                    404 idempotency
```

The final cell derives, from the observed status codes, **which gate produces the
403** (privilege vs config) — answering the open question rather than assuming it.

## Setup note (important)

```
A grant can't be cleanly REVOKED mid-test, so the runner uses TWO catalogs:
  view-test-grant     — CATALOG_MANAGE_CONTENT granted
  view-test-nogrant   — NOT granted
Each case runs against the right one. Both are cleaned up (verified) at the end.
```

## Prerequisites & how to run

```
Files:   polaris_test_utils.py, minio_rest.py, config/  (same folder)
Packages: pip install requests pyyaml opensearch-py
Run:     setup cell (🟢/🔴 banner) → force_delete_catalog def → matrix setup →
         C1–C8 → truth table → cleanup.
Switch:  change init_env("dev") → init_env("prod") in setup, re-run that cell.

⚠️  Storage via MinIO REST API (minio_rest.py) — NO s3fs. No hardcoded creds.
```

---


In [1]:
# ── Setup ─────────────────────────────────────────────────────────────────
# Environment switch HERE. Banner shows 🟢 DEV / 🔴 PROD. Change "dev" → "prod"
# and re-run THIS cell to switch (no kernel restart needed).
from polaris_test_utils import *
init_env("dev")

import uuid, time
tok = root_token()

# Map friendly kwarg names → the EXACT Polaris property keys.
# (Property names mix hyphens AND dots — a blind _→- replace produces the wrong
#  key, e.g. "drop-with-purge-enabled" instead of "drop-with-purge.enabled",
#  which Polaris rejects with 400. That silent failure caused earlier wrong results.)
_CONFIG_KEYS = {
    "drop_with_purge_enabled":       "polaris.config.drop-with-purge.enabled",
    "purge_view_metadata_on_drop":   "polaris.config.purge-view-metadata-on-drop",
}

def set_catalog_config(catalog, _retries=5, **flags):
    """Merge config flags into a catalog AND VERIFY they applied.
    Uses the exact property keys, checks the PUT response, retries on 409
    entityVersion conflict, and confirms the values changed by reading back."""
    want = {}
    for k, v in flags.items():
        if k not in _CONFIG_KEYS:
            raise KeyError(f"unknown config flag '{k}'. Known: {list(_CONFIG_KEYS)}")
        want[_CONFIG_KEYS[k]] = v
    for attempt in range(_retries):
        cat = requests.get(f"{BASE_MGMT}/catalogs/{catalog}", headers=h(tok)).json()
        merged = {**cat.get("properties", {}), **want}
        r = requests.put(f"{BASE_MGMT}/catalogs/{catalog}", headers=h(tok),
                         json={"currentEntityVersion": cat["entityVersion"], "properties": merged})
        if r.status_code in (200, 204):
            now = requests.get(f"{BASE_MGMT}/catalogs/{catalog}", headers=h(tok)).json().get("properties", {})
            if all(now.get(k) == v for k, v in want.items()):
                return True
        elif r.status_code == 409:
            time.sleep(0.2); continue
        else:
            raise RuntimeError(f"set_catalog_config PUT failed [{r.status_code}]: {r.text[:200]}")
    now = requests.get(f"{BASE_MGMT}/catalogs/{catalog}", headers=h(tok)).json().get("properties", {})
    if not all(now.get(k) == v for k, v in want.items()):
        raise RuntimeError(f"set_catalog_config could not apply {want} to {catalog} "
                           f"(now: { {k: now.get(k) for k in want} })")
    return True

print("✅ setup ready (correct property keys + verified writes)")


🟢 DEV   POLARIS_ENV=dev   (Polaris 1.3.0)
   Polaris: http://192.168.139.2:8181
   MinIO:   http://192.168.139.2:9000  bucket=data-catalog-bucket
   OpenSearch: localhost:9200  index=k8s-logs-*
   purge_deletes_files=False
🟢 DEV   POLARIS_ENV=dev   (Polaris 1.3.0)
   Polaris: http://192.168.139.2:8181
   MinIO:   http://192.168.139.2:9000  bucket=data-catalog-bucket
   OpenSearch: localhost:9200  index=k8s-logs-*
   purge_deletes_files=False
✅ setup ready (correct property keys + verified writes)


In [2]:
# HARDENED cleanup — removes a catalog completely and VERIFIES it's gone.
# Key fixes:
#  • drop views/tables WITHOUT ?purgeRequested (avoids the purge-coupling that
#    can make a view drop fail and leave the namespace non-empty)
#  • set purge-view-metadata=false BEFORE dropping views (so drop is metadata-only)
#  • verify namespace is empty before deleting it; verify catalog gone at the end
# A stale catalog with old config flags is what corrupted earlier test runs.

def _ns_name(ns):
    return ns[0] if isinstance(ns, list) else ns

def force_delete_catalog(cat):
    if requests.get(f"{BASE_MGMT}/catalogs/{cat}", headers=h(tok)).status_code == 404:
        return True
    # make teardown safe: purge OFF for views (metadata-only drop), purge ON allowed
    set_catalog_config(cat, drop_with_purge_enabled="true", purge_view_metadata_on_drop="false")
    grant_privilege(catalog=cat, cr="catalog_admin", privilege="CATALOG_MANAGE_CONTENT", token=tok)

    nss = requests.get(f"{BASE_CAT}/{cat}/namespaces", headers=h(tok)).json().get("namespaces", [])
    # deepest namespaces first
    nss = sorted(nss, key=lambda n: len(n if isinstance(n, list) else [n]), reverse=True)
    for ns in nss:
        nsn = _ns_name(ns)
        # drop all tables (plain drop — no purge; storage swept separately)
        for t in requests.get(f"{BASE_CAT}/{cat}/namespaces/{nsn}/tables", headers=h(tok)).json().get("identifiers", []):
            rt = requests.delete(f"{BASE_CAT}/{cat}/namespaces/{nsn}/tables/{t['name']}", headers=h(tok))
            if rt.status_code not in (200,204):
                print(f"   ⚠️ table {t['name']} drop: {rt.status_code} {rt.text[:80]}")
        # drop all views (plain drop — no purge)
        for v in requests.get(f"{BASE_CAT}/{cat}/namespaces/{nsn}/views", headers=h(tok)).json().get("identifiers", []):
            rv = requests.delete(f"{BASE_CAT}/{cat}/namespaces/{nsn}/views/{v['name']}", headers=h(tok))
            if rv.status_code not in (200,204):
                print(f"   ⚠️ view {v['name']} drop: {rv.status_code} {rv.text[:80]}")
        # confirm empty, then delete namespace
        tleft = requests.get(f"{BASE_CAT}/{cat}/namespaces/{nsn}/tables", headers=h(tok)).json().get("identifiers", [])
        vleft = requests.get(f"{BASE_CAT}/{cat}/namespaces/{nsn}/views", headers=h(tok)).json().get("identifiers", [])
        if tleft or vleft:
            print(f"   ❌ namespace {nsn} still has tables={len(tleft)} views={len(vleft)} — cannot delete")
            continue
        rns = requests.delete(f"{BASE_CAT}/{cat}/namespaces/{nsn}", headers=h(tok))
        if rns.status_code not in (200,204):
            print(f"   ⚠️ namespace {nsn} delete: {rns.status_code} {rns.text[:100]}")

    # drop non-default catalog-roles (a catalog with extra roles is "not empty")
    for cr in requests.get(f"{BASE_MGMT}/catalogs/{cat}/catalog-roles", headers=h(tok)).json().get("roles", []):
        crn = cr["name"] if isinstance(cr, dict) else cr
        if crn != "catalog_admin":
            rcr = requests.delete(f"{BASE_MGMT}/catalogs/{cat}/catalog-roles/{crn}", headers=h(tok))
            if rcr.status_code not in (200,204):
                print(f"   ⚠️ catalog-role {crn} delete: {rcr.status_code} {rcr.text[:80]}")
    rc = requests.delete(f"{BASE_MGMT}/catalogs/{cat}?purgeRequested=true", headers=h(tok))
    mc.delete_prefix(f"{cat}/", min_size=0)
    still = requests.get(f"{BASE_MGMT}/catalogs/{cat}", headers=h(tok)).status_code != 404
    if still:
        print(f"   ❌ catalog {cat} STILL EXISTS (delete status {rc.status_code}): {rc.text[:150]}")
        # show what's blocking
        leftover = requests.get(f"{BASE_CAT}/{cat}/namespaces", headers=h(tok)).json().get("namespaces", [])
        print(f"      remaining namespaces: {leftover}")
        return False
    return True

print("✅ force_delete_catalog ready")


✅ force_delete_catalog ready


## Test matrix — 8 cases across config × privilege × purgeRequested

We map which gate controls a view drop. Four dimensions interact:

```
purge-view-metadata-on-drop   catalog config (true/false)
drop-with-purge.enabled       catalog config (true/false)
?purgeRequested=              request param  (true/false/absent)
CATALOG_MANAGE_CONTENT        principal privilege (granted/not)
```

Because a grant can't be cleanly revoked mid-test, the runner uses **two catalogs**:
one WITH the grant and one WITHOUT. Each case runs against the right one.

```
C1  purge-meta=true  drop-purge=false  grant=NO   purgeReq=absent   ← original 403?
C2  purge-meta=true  drop-purge=false  grant=YES  purgeReq=absent   isolates privilege
C3  purge-meta=true  drop-purge=true   grant=NO   purgeReq=absent   isolates config
C4  purge-meta=true  drop-purge=true   grant=YES  purgeReq=absent   everything on
C5  purge-meta=true  drop-purge=true   grant=YES  purgeReq=false    explicit false
C6  purge-meta=false drop-purge=true   grant=YES  purgeReq=true     param vs config
C7  purge-meta=false drop-purge=true   grant=YES  purgeReq=absent   metadata-only baseline
C8  (drop an already-dropped view)                                  404 idempotency
```


In [3]:
# Per-case catalogs: each case creates its OWN dedicated catalog with the config
# baked in at create time and VERIFIED (see fresh_catalog_for_case). This replaces
# the earlier "two shared catalogs + flip properties" design, which was unreliable
# because property flips could silently fail. NS name used by all cases:
NS = "vp-ns"
print("✅ using per-case dedicated catalogs (config verified at creation)")


✅ using per-case dedicated catalogs (config verified at creation)


### Verify privileges BEFORE running the matrix

Confirms the actual identity + grants we're testing with, so the results are
interpretable. Specifically: which principal the token belongs to, and what
grants exist on each catalog's `catalog_admin` role. If the token is the **root
principal**, it may bypass privilege checks — which would explain why a
privilege-gated 403 doesn't appear for any matrix case.


In [4]:
# Verify WHO we authenticate as (privilege context for the whole test).
import base64, json as _json
def token_principal(t):
    try:
        p = t.split(".")[1]; p += "="*(-len(p)%4)
        c = _json.loads(base64.urlsafe_b64decode(p))
        return c.get("sub") or c.get("principal") or c.get("client_id")
    except Exception as e:
        return f"(decode failed: {e})"
print("Token principal:", token_principal(tok), "| ROOT_CLIENT:", ROOT_CLIENT)
pr = requests.get(f"{BASE_MGMT}/principal-roles", headers=h(tok))
if pr.status_code==200:
    print("Principal-roles:", [r["name"] for r in pr.json().get("roles", [])])
print("→ root maps to service_admin (bypasses catalog-role privilege checks),")
print("  which is why C9 uses a separate NON-root principal to test privilege.")


Token principal: root | ROOT_CLIENT: root
Principal-roles: ['service_admin']
→ root maps to service_admin (bypasses catalog-role privilege checks),
  which is why C9 uses a separate NON-root principal to test privilege.


In [5]:
results = {}
import re as _re

def _ns(): return "vp-ns"

def make_view(cat, name):
    return requests.post(f"{BASE_CAT}/{cat}/namespaces/{_ns()}/views", headers=h(tok),
        json={"name": name, "default-namespace": [_ns()],
              "schema":{"type":"struct","schema-id":0,
                        "fields":[{"id":1,"name":"id","type":"long","required":True}]},
              "view-version":{"version-id":1,"timestamp-ms":int(time.time()*1000),"schema-id":0,
                              "default-namespace":[_ns()],"summary":{"engine":"spark"},
                              "representations":[{"type":"sql","sql":"SELECT 1 AS id","dialect":"spark"}]},
              "location": f"s3a://{BUCKET}/{cat}/{_ns()}/{name}/"})

def cat_files(cat, name):
    return mc.list(f"{cat}/{_ns()}/{name}/")

def fresh_catalog_for_case(case, purge_meta, drop_purge, grant):
    """Create a DEDICATED catalog for this case with config baked in at CREATE
    time, then VERIFY the properties actually took. Each case gets its own
    catalog so no property-flip / reuse can contaminate it."""
    cat = f"vptest-{case.lower()}"
    force_delete_catalog(cat)
    # bake config into create (create_catalog merges these into properties)
    create_catalog(cat, token=tok, properties={
        "polaris.config.purge-view-metadata-on-drop": str(purge_meta).lower(),
        "polaris.config.drop-with-purge.enabled":     str(drop_purge).lower()})
    # VERIFY the catalog really has the intended config
    props = requests.get(f"{BASE_MGMT}/catalogs/{cat}", headers=h(tok)).json().get("properties", {})
    got_meta  = props.get("polaris.config.purge-view-metadata-on-drop")
    got_purge = props.get("polaris.config.drop-with-purge.enabled")
    assert got_meta == str(purge_meta).lower(), f"{cat}: purge-meta={got_meta} (wanted {purge_meta})"
    assert got_purge == str(drop_purge).lower(), f"{cat}: drop-purge={got_purge} (wanted {drop_purge})"
    # grant CATALOG_MANAGE_CONTENT only when grant=True
    if grant:
        grant_privilege(catalog=cat, cr="catalog_admin", privilege="CATALOG_MANAGE_CONTENT", token=tok)
    create_namespace(catalog=cat, ns=_ns(), token=tok)
    return cat, got_meta, got_purge

def run_case(case, purge_meta, drop_purge, grant, purge_requested):
    """One case = one dedicated catalog (config verified) → create view → drop →
    measure status, file outcome, AND whether a delete/cleanup was logged."""
    cat, vmeta, vpurge = fresh_catalog_for_case(case, purge_meta, drop_purge, grant)
    vname = f"{case}-{uuid.uuid4().hex[:6]}"
    make_view(cat, vname)
    before = len(cat_files(cat, vname))

    url = f"{BASE_CAT}/{cat}/namespaces/{_ns()}/views/{vname}"
    if purge_requested is True:    url += "?purgeRequested=true"
    elif purge_requested is False: url += "?purgeRequested=false"
    drop_t0 = time.time()
    r = requests.delete(url, headers=h(tok))

    gone = requests.get(f"{BASE_CAT}/{cat}/namespaces/{_ns()}/views/{vname}",
                        headers=h(tok)).status_code == 404
    time.sleep(2)
    after = len(cat_files(cat, vname))

    pr = {True:"true", False:"false", None:"absent"}[purge_requested]
    results[case] = {"cat":cat, "vname":vname, "purge_meta":purge_meta, "drop_purge":drop_purge,
                     "grant":grant, "purge_req":pr, "status":r.status_code, "gone":gone,
                     "before":before, "after":after, "cfg_verified":(vmeta,vpurge),
                     "err": r.text[:160] if r.status_code>=400 else "", "drop_t0":drop_t0}
    fo = ("orphaned" if after>=before and before>0 else
          "deleted"  if after==0 and before>0 else f"{before}->{after}")
    print(f"[{case}] cat={cat} (cfg verified meta={vmeta} purge={vpurge}) grant={grant} req={pr}")
    print(f"   status={r.status_code}  gone={gone}  file={fo}"
          + (f"  ERR: {r.text[:90]}" if r.status_code>=400 else ""))
    return results[case]

print("✅ run_case ready (per-case dedicated catalog, config verified at create)")


✅ run_case ready (per-case dedicated catalog, config verified at create)


In [6]:
# C1 — purge-meta=true, drop-purge=false, NO grant, purgeReq absent  (original 403?)
run_case("C1", purge_meta=True,  drop_purge=False, grant=False, purge_requested=None)

[C1] cat=vptest-c1 (cfg verified meta=true purge=false) grant=False req=absent
   status=403  gone=False  file=orphaned  ERR: {"error":{"message":"Unable to purge entity: C1-e1adf7. To enable this feature, set the Po


{'cat': 'vptest-c1',
 'vname': 'C1-e1adf7',
 'purge_meta': True,
 'drop_purge': False,
 'grant': False,
 'purge_req': 'absent',
 'status': 403,
 'gone': False,
 'before': 1,
 'after': 1,
 'cfg_verified': ('true', 'false'),
 'err': '{"error":{"message":"Unable to purge entity: C1-e1adf7. To enable this feature, set the Polaris configuration DROP_WITH_PURGE_ENABLED or the catalog configurati',
 'drop_t0': 1782452123.445738}

In [7]:
# C2 — purge-meta=true, drop-purge=false, WITH grant  (isolates privilege)
run_case("C2", purge_meta=True,  drop_purge=False, grant=True,  purge_requested=None)

[C2] cat=vptest-c2 (cfg verified meta=true purge=false) grant=True req=absent
   status=403  gone=False  file=orphaned  ERR: {"error":{"message":"Unable to purge entity: C2-bb559d. To enable this feature, set the Po


{'cat': 'vptest-c2',
 'vname': 'C2-bb559d',
 'purge_meta': True,
 'drop_purge': False,
 'grant': True,
 'purge_req': 'absent',
 'status': 403,
 'gone': False,
 'before': 1,
 'after': 1,
 'cfg_verified': ('true', 'false'),
 'err': '{"error":{"message":"Unable to purge entity: C2-bb559d. To enable this feature, set the Polaris configuration DROP_WITH_PURGE_ENABLED or the catalog configurati',
 'drop_t0': 1782452128.083391}

In [8]:
# C3 — purge-meta=true, drop-purge=true, NO grant  (isolates config)
run_case("C3", purge_meta=True,  drop_purge=True,  grant=False, purge_requested=None)

[C3] cat=vptest-c3 (cfg verified meta=true purge=true) grant=False req=absent
   status=204  gone=True  file=orphaned


{'cat': 'vptest-c3',
 'vname': 'C3-679ef1',
 'purge_meta': True,
 'drop_purge': True,
 'grant': False,
 'purge_req': 'absent',
 'status': 204,
 'gone': True,
 'before': 1,
 'after': 1,
 'cfg_verified': ('true', 'true'),
 'err': '',
 'drop_t0': 1782452131.1707299}

In [9]:
# C4 — purge-meta=true, drop-purge=true, WITH grant  (everything on)
run_case("C4", purge_meta=True,  drop_purge=True,  grant=True,  purge_requested=None)

[C4] cat=vptest-c4 (cfg verified meta=true purge=true) grant=True req=absent
   status=204  gone=True  file=orphaned


{'cat': 'vptest-c4',
 'vname': 'C4-c1c188',
 'purge_meta': True,
 'drop_purge': True,
 'grant': True,
 'purge_req': 'absent',
 'status': 204,
 'gone': True,
 'before': 1,
 'after': 1,
 'cfg_verified': ('true', 'true'),
 'err': '',
 'drop_t0': 1782452140.5609548}

In [10]:
# C5 — explicit purgeRequested=false (vs absent in C4)
run_case("C5", purge_meta=True,  drop_purge=True,  grant=True,  purge_requested=False)

[C5] cat=vptest-c5 (cfg verified meta=true purge=true) grant=True req=false
   status=204  gone=True  file=orphaned


{'cat': 'vptest-c5',
 'vname': 'C5-e8e6b2',
 'purge_meta': True,
 'drop_purge': True,
 'grant': True,
 'purge_req': 'false',
 'status': 204,
 'gone': True,
 'before': 1,
 'after': 1,
 'cfg_verified': ('true', 'true'),
 'err': '',
 'drop_t0': 1782452143.7806249}

In [11]:
# C6 — purgeRequested=true but purge-meta=false  (param vs config conflict)
run_case("C6", purge_meta=False, drop_purge=True,  grant=True,  purge_requested=True)

[C6] cat=vptest-c6 (cfg verified meta=false purge=true) grant=True req=true
   status=204  gone=True  file=orphaned


{'cat': 'vptest-c6',
 'vname': 'C6-a3fd40',
 'purge_meta': False,
 'drop_purge': True,
 'grant': True,
 'purge_req': 'true',
 'status': 204,
 'gone': True,
 'before': 1,
 'after': 1,
 'cfg_verified': ('false', 'true'),
 'err': '',
 'drop_t0': 1782452146.859009}

In [12]:
# C7 — purge-meta=false  (metadata-only baseline)
run_case("C7", purge_meta=False, drop_purge=True,  grant=True,  purge_requested=None)

[C7] cat=vptest-c7 (cfg verified meta=false purge=true) grant=True req=absent
   status=204  gone=True  file=orphaned


{'cat': 'vptest-c7',
 'vname': 'C7-95d99d',
 'purge_meta': False,
 'drop_purge': True,
 'grant': True,
 'purge_req': 'absent',
 'status': 204,
 'gone': True,
 'before': 1,
 'after': 1,
 'cfg_verified': ('false', 'true'),
 'err': '',
 'drop_t0': 1782452149.709654}

In [13]:
# C8 — idempotency: drop an already-dropped view (expect 404 on second drop)
cat, _, _ = fresh_catalog_for_case("C8", purge_meta=False, drop_purge=True, grant=True)
vname = f"C8-{uuid.uuid4().hex[:6]}"
make_view(cat, vname)
r1 = requests.delete(f"{BASE_CAT}/{cat}/namespaces/{_ns()}/views/{vname}", headers=h(tok))
r2 = requests.delete(f"{BASE_CAT}/{cat}/namespaces/{_ns()}/views/{vname}", headers=h(tok))
results["C8"] = {"cat":cat,"vname":vname,"purge_meta":"-","drop_purge":"-","grant":True,
                 "purge_req":"absent","status":r2.status_code,"gone":True,
                 "before":0,"after":0,"err":r2.text[:140] if r2.status_code>=400 else ""}
print(f"[C8] first drop={r1.status_code}, second drop={r2.status_code} (expect 404)")


[C8] first drop=204, second drop=404 (expect 404)


## C9 — the real privilege test: a NON-ROOT principal

C1–C8 all ran as **root** (service_admin), which bypasses catalog-role privilege
checks — so they couldn't trigger a privilege 403. C9 creates a fresh principal
that has only enough rights to SEE the catalog but **NOT** `CATALOG_MANAGE_CONTENT`,
authenticates as that principal, and tries to drop a view.

```
If C9 → 403  →  the privilege gate is REAL; root was bypassing it.
                The original 403 was a non-root principal lacking the privilege.
If C9 → 204  →  view drop needs no special privilege even for non-root.
```


In [14]:
# C9 — create a limited non-root principal and drop a view as that principal
P9   = "view-limited-principal"
PR9  = "view-limited-prole"
CR9  = "view-limited-crole"
cat, _, _ = fresh_catalog_for_case("C9", purge_meta=True, drop_purge=True, grant=False)

# clean any leftovers from a prior run
requests.delete(f"{BASE_MGMT}/principals/{P9}", headers=h(tok))
requests.delete(f"{BASE_MGMT}/principal-roles/{PR9}", headers=h(tok))

# 1. create principal → capture its client credentials
rp = create_principal(P9, token=tok)
creds = rp.json()
client_id     = creds.get("credentials", {}).get("clientId") or creds.get("clientId")
client_secret = creds.get("credentials", {}).get("clientSecret") or creds.get("clientSecret")
print(f"created principal {P9}: client_id={client_id}")

# 2. principal-role + catalog-role, give the catalog-role ONLY limited privileges
create_principal_role(PR9, token=tok)
create_catalog_role(catalog=cat, name=CR9, token=tok)
assign_catalog_role_to_principal_role(catalog=cat, pr=PR9, cr=CR9, token=tok)
assign_principal_role_to_principal(principal=P9, pr=PR9, token=tok)
# grant ONLY enough to use the catalog + drop a view, but NOT manage-content/purge
for priv in ["CATALOG_MANAGE_METADATA", "TABLE_LIST", "VIEW_LIST", "VIEW_CREATE", "VIEW_DROP"]:
    grant_privilege(catalog=cat, cr=CR9, privilege=priv, token=tok)
print(f"granted limited privileges to {CR9} (NO CATALOG_MANAGE_CONTENT)")

# 3. get a token AS the limited principal
def principal_token(cid, csec):
    r = get_token(client_id=cid, client_secret=csec)
    return r.json().get("access_token") if r.status_code == 200 else None

ltok = principal_token(client_id, client_secret)
print(f"limited-principal token obtained: {ltok is not None}")

# 4. create a view (as root, to be sure it exists), then DROP as the limited principal
set_catalog_config(cat, purge_view_metadata_on_drop="true", drop_with_purge_enabled="true")
vname = f"C9-{uuid.uuid4().hex[:6]}"
make_view(cat, vname)   # created with root

if ltok:
    rh = {"Authorization": f"Bearer {ltok}", "Polaris-Realm": REALM, "Content-Type": "application/json"}
    r = requests.delete(f"{BASE_CAT}/{cat}/namespaces/{NS}/views/{vname}", headers=rh)
    print(f"\n[C9] DROP as NON-ROOT (no CATALOG_MANAGE_CONTENT): {r.status_code}")
    print(f"     {r.text[:200]}")
    results["C9"] = {"purge_meta":True,"drop_purge":True,"grant":"non-root-limited",
                     "purge_req":"absent","status":r.status_code,
                     "gone": requests.get(f"{BASE_CAT}/{cat}/namespaces/{NS}/views/{vname}", headers=h(tok)).status_code==404,
                     "before":1,"after":1,"err":r.text[:140] if r.status_code>=400 else ""}
    print()
    if r.status_code == 403:
        print("✅ C9 = 403 → PRIVILEGE GATE CONFIRMED.")
        print("   A non-root principal without CATALOG_MANAGE_CONTENT CANNOT drop the view.")
        print("   → root (service_admin) was bypassing this gate in C1–C8.")
        print("   → the original 403 was a non-root principal lacking the privilege.")
    elif r.status_code in (200,204):
        print("ℹ️ C9 = 204 → even a limited non-root principal can drop a view.")
        print("   → VIEW_DROP alone suffices; the 403 came from something else.")
    else:
        print(f"ℹ️ C9 = {r.status_code} — inspect the error body above.")
else:
    print("⚠️ could not obtain limited-principal token; check principal creation")


created principal view-limited-principal: client_id=097c13dc1bcd0dcd
granted limited privileges to view-limited-crole (NO CATALOG_MANAGE_CONTENT)
limited-principal token obtained: True

[C9] DROP as NON-ROOT (no CATALOG_MANAGE_CONTENT): 204
     

ℹ️ C9 = 204 → even a limited non-root principal can drop a view.
   → VIEW_DROP alone suffices; the 403 came from something else.


In [15]:
# C9 cleanup — remove the limited principal, its roles, AND its catalog-role
requests.delete(f"{BASE_MGMT}/principals/{P9}", headers=h(tok))
requests.delete(f"{BASE_MGMT}/principal-roles/{PR9}", headers=h(tok))
requests.delete(f"{BASE_MGMT}/catalogs/{cat}/catalog-roles/{CR9}", headers=h(tok))  # ← was leaking
print("✅ C9 principal + principal-role + catalog-role removed")


✅ C9 principal + principal-role + catalog-role removed


## Deletion-request log check (does Polaris ATTEMPT a delete?)

`mc.list` only tells us whether files REMAIN — it cannot tell us whether Polaris
*attempted* to delete them. To know if a drop actually triggered a deletion/cleanup,
we query the Polaris logs in OpenSearch for each dropped view's name.

```
For each case, look for either:
  • "Scheduled cleanup task ... for ..."   (a cleanup task was created)
  • view-name appearing in a delete/drop log line
A 204 with NO cleanup-task log → the drop was metadata-only (no delete attempted).
A 204 WITH a cleanup-task log → a delete WAS attempted (and orphaned, per #379).
```
> Requires the company log OpenSearch (the env banner shows the host).


In [16]:
# For each completed case, check the Polaris logs for a deletion/cleanup attempt
# tied to that view's name. Analyzer-proof: pull broadly, substring-filter in Python.
def deletion_logs_for(view_name, minutes=20):
    """Return Polaris log messages mentioning this view name (any logger)."""
    try:
        hits = search_logs_raw({
            "query": {"bool": {"must": [
                {"range": {"@timestamp": {"gte": f"now-{minutes}m"}}},
                {"term": {"kubernetes.container_name.keyword": POLARIS_CONTAINER}}]}},
            "sort": [{"@timestamp": {"order": "asc"}}]
        }, size=400)
    except Exception as e:
        print(f"   (log query failed: {str(e)[:80]})")
        return []
    msgs = [hh["_source"].get("message","") for hh in hits]
    # substring-filter for this view name (hyphenated → analyzer-proof in Python)
    return [m for m in msgs if view_name in m]

print("Per-case deletion-attempt evidence from Polaris logs:\n")
for case in ["C1","C2","C3","C4","C5","C6","C7"]:
    r = results.get(case)
    if not r:
        continue
    vn = r["vname"]
    lines = deletion_logs_for(vn)
    scheduled = [m for m in lines if "Scheduled cleanup task" in m or "cleanup task" in m.lower()]
    print(f"[{case}] view={vn}  status={r['status']}  req={r['purge_req']}")
    print(f"    log lines mentioning view: {len(lines)}   cleanup-task scheduled: {len(scheduled)}")
    for m in scheduled[:2]:
        print(f"      └─ {m[:110]}")
    # interpretation
    if r["status"] == 204:
        if scheduled:
            print("    → 204 + cleanup task scheduled → Polaris ATTEMPTED deletion (orphaned per #379)")
        else:
            print("    → 204 + NO cleanup task → metadata-only drop (no deletion attempted)")
    elif r["status"] == 403:
        print("    → 403 → drop blocked before any deletion attempt")
    print()


Per-case deletion-attempt evidence from Polaris logs:

[C1] view=C1-e1adf7  status=403  req=absent
    log lines mentioning view: 0   cleanup-task scheduled: 0
    → 403 → drop blocked before any deletion attempt

[C2] view=C2-bb559d  status=403  req=absent
    log lines mentioning view: 0   cleanup-task scheduled: 0
    → 403 → drop blocked before any deletion attempt

[C3] view=C3-679ef1  status=204  req=absent
    log lines mentioning view: 0   cleanup-task scheduled: 0
    → 204 + NO cleanup task → metadata-only drop (no deletion attempted)

[C4] view=C4-c1c188  status=204  req=absent
    log lines mentioning view: 0   cleanup-task scheduled: 0
    → 204 + NO cleanup task → metadata-only drop (no deletion attempted)

[C5] view=C5-e8e6b2  status=204  req=false
    log lines mentioning view: 0   cleanup-task scheduled: 0
    → 204 + NO cleanup task → metadata-only drop (no deletion attempted)

[C6] view=C6-a3fd40  status=204  req=true
    log lines mentioning view: 0   cleanup-task s

## Truth table — observed results

In [17]:
print("="*78)
print(f"VIEW DROP — case matrix (Polaris {POLARIS_VERSION})")
print("="*78)
print(f"{'case':5}{'purge-meta':11}{'drop-purge':11}{'grant':7}{'purgeReq':9}{'status':7}{'file':10}{'gone'}")
print("-"*78)
order = ["C1","C2","C3","C4","C5","C6","C7","C8","C9"]
for cse in order:
    r = results.get(cse)
    if not r:
        print(f"{cse:5}(not run)"); continue
    fo = ("orphaned" if r["after"]>=r["before"] and r["before"]>0 else
          "deleted"  if r["after"]==0 and r["before"]>0 else "-")
    print(f"{cse:5}{str(r['purge_meta']):11}{str(r['drop_purge']):11}"
          f"{str(r['grant']):7}{r['purge_req']:9}{str(r['status']):7}{fo:10}{r['gone']}")
    if r["err"]:
        print(f"      └─ {r['err'][:88]}")

def st(c): return results.get(c,{}).get("status")

print()
print("="*78)
print("WHAT EACH GATE DOES (derived from the cases above)")
print("="*78)

print("\n1. CONFIG gate — drop-with-purge.enabled (the decisive comparison):")
print(f"   C1/C2 drop-purge=FALSE → {st('C1')}/{st('C2')}")
print(f"   C3/C4 drop-purge=TRUE  → {st('C3')}/{st('C4')}")
if st("C1")==403 and st("C3") in (200,204):
    print("   → ✅ drop-with-purge.enabled=false BLOCKS the view drop (403).")
    print("        Setting it true allows the drop (204). This is the CONFIG gate.")
    print("        The 403 error text ('To enable this feature, set the ... config')")
    print("        confirms it is a CONFIG block, not a privilege block.")

print("\n2. PRIVILEGE — does CATALOG_MANAGE_CONTENT matter? (compare grant vs no-grant):")
print(f"   C1 no-grant={st('C1')}  vs  C2 grant={st('C2')}   (both drop-purge=false)")
print(f"   C3 no-grant={st('C3')}  vs  C4 grant={st('C4')}   (both drop-purge=true)")
if st("C1")==st("C2") and st("C3")==st("C4"):
    print("   → grant vs no-grant makes NO difference at the same config.")
    print("     → privilege does NOT gate the view drop (root bypasses anyway).")

print("\n3. PRIVILEGE — decisive non-root test (C9):")
c9 = results.get("C9", {})
if c9:
    print(f"   C9 (non-root, no CATALOG_MANAGE_CONTENT, drop-purge=TRUE) → {c9.get('status')}")
    if c9.get("status") in (200,204):
        print("   → even a limited non-root principal can drop (when config allows).")
        print("     → CONFIRMS: view drop is gated by CONFIG, not privilege.")
    elif c9.get("status")==403:
        print("   → non-root blocked → privilege IS an additional gate.")

print("\n4. purgeRequested param:")
print(f"   C4 absent={st('C4')}  C5 explicit-false={st('C5')}  → "
      + ("no difference" if st('C4')==st('C5') else "differs"))
print(f"   C6 (purgeReq=true, purge-meta=false)={st('C6')}")
print("   (see the deletion-request log check above for whether a delete was ATTEMPTED)")

print("\n5. Baseline + idempotency:")
print(f"   C7 metadata-only (purge-meta=false)={st('C7')}   C8 double-drop={st('C8')} (expect 404)")

print("\nNOTE: 'orphaned' = drop succeeded but metadata file NOT deleted")
print("      (no-STS cleanup-task failure, Polaris #379).")


VIEW DROP — case matrix (Polaris 1.3.0)
case purge-meta drop-purge grant  purgeReq status file      gone
------------------------------------------------------------------------------
C1   True       False      False  absent   403    orphaned  False
      └─ {"error":{"message":"Unable to purge entity: C1-e1adf7. To enable this feature, set the 
C2   True       False      True   absent   403    orphaned  False
      └─ {"error":{"message":"Unable to purge entity: C2-bb559d. To enable this feature, set the 
C3   True       True       False  absent   204    orphaned  True
C4   True       True       True   absent   204    orphaned  True
C5   True       True       True   false    204    orphaned  True
C6   False      True       True   true     204    orphaned  True
C7   False      True       True   absent   204    orphaned  True
C8   -          -          True   absent   404    -         True
      └─ {"error":{"message":"View does not exist: vp-ns.C8-86bd90","type":"NoSuchViewException",


## Cleanup

In [18]:
# Cleanup — remove every per-case catalog created by this run
for case in ["C1","C2","C3","C4","C5","C6","C7","C8","C9"]:
    force_delete_catalog(f"vptest-{case.lower()}")
print("✅ cleaned up all per-case catalogs")


✅ cleaned up all per-case catalogs
